In [1]:
import pandas as pd
import numpy as np
import os

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [4]:

# =====================================================
# 1. LOAD DATASET
# =====================================================

print("Loading dataset...")

df = pd.read_csv("Datasets/data.csv")

print("Total samples:", len(df))
print("Total features:", len(df.columns) - 1)

print("Dataset  Loaded...")

Loading dataset...
Total samples: 246945
Total features: 377
Dataset  Loaded...


In [5]:
# =====================================================
# 2. SPLIT FEATURES AND TARGET
# =====================================================

X = df.drop("diseases", axis=1)
y = df["diseases"]

# Convert to float32 to reduce memory usage
X = X.astype("float32")

In [6]:
# =====================================================
# 1. TRAIN / TEST SPLIT
# =====================================================

print("Splitting dataset...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Splitting dataset...
Training shape: (197556, 377)
Testing shape: (49389, 377)


In [27]:
# =====================================================
# 2. CREATE POOLS
# =====================================================

train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)

In [28]:
# =====================================================
# 3. MODEL CONFIGURATION
# =====================================================

print("\nInitializing CatBoost model...")


print("Training on GPU...")

model = CatBoostClassifier(
    iterations=1100,
    learning_rate=0.03,
    depth=8,

    loss_function="MultiClass",
    eval_metric="Accuracy",

    # task_type="GPU",      # 🔥 ENABLE GPU ->Kaggle
    devices="0",          # Use first GPU

    auto_class_weights="Balanced",

    early_stopping_rounds=100,

    verbose=100
)



Initializing CatBoost model...
Training on GPU...


In [12]:

# =====================================================
# 4. TRAIN MODEL
# =====================================================

print("\nTraining model...")

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)



print("Training completed!")



Training model...


KeyboardInterrupt: 

In [29]:
# =====================================================
# 5. EVALUATE MODEL
# =====================================================

print("\nEvaluating model...")

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)

print("Test Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, preds))



Evaluating model...


CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

In [30]:

# =====================================================
# 6. PREDICTION CONFIDENCE CHECK
# =====================================================

print("\nChecking prediction confidence...")

probs = model.predict_proba(X_test)

print("Example prediction probabilities:")
print(probs[:5])


Checking prediction confidence...


CatBoostError: There is no trained model to use predict_proba(). Use fit() to train model. Then use this method.

In [31]:
# =====================================================
# 9. SAVE MODEL
# =====================================================


model.save_model("models/disease_prediction_model.cbm")

print("Model saved successfully!")


CatBoostError: There is no trained model to use save_model(). Use fit() to train model. Then use this method.

In [ ]:
import pickle


# Save symptom list
symptom_columns = X.columns.tolist()
pickle.dump(symptom_columns, open("models/symptom_columns.pkl","wb"))



All artifacts saved!


In [32]:
from catboost import CatBoostClassifier

model = CatBoostClassifier()
model.load_model("models/disease_prediction_model.cbm")

CatBoostClassifier(auto_class_weights='Balanced', depth=8, devices='0', eval_metric='Accuracy', iterations=1100, learning_rate=0.03, loss_function='MultiClass', od_wait=100, task_type='GPU', use_best_model=True, verbose=100)

In [33]:
# Save disease labels
pickle.dump(model.classes_, open("models/disease_classes.pkl","wb"))

print("All artifacts saved!")

All artifacts saved!
